# مرور قابل بازتولید تحلیل پایان‌نامه

این Notebook خروجی‌های ساخته‌شده با `python run_all.py` را مرحله‌به‌مرحله مرور می‌کند. اجرای مدل‌ها در اسکریپت اصلی انجام می‌شود و این فایل برای کنترل مسیر داده تا نتیجه است.


In [1]:
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "results").is_dir():
    ROOT = ROOT.parent
RESULTS = ROOT / "results"
print(f"project root: {ROOT}")


project root: /workspace/scratch/d4cc90b4c144/hr-analytics-ml


## ۱. شناسنامه چهار دیتاست


In [2]:
inventory = pd.read_csv(RESULTS / "data" / "dataset_inventory.csv")
print(inventory[["Dataset", "Display_Name", "Role", "Rows", "Raw_Columns", "Target", "SHA256"]].to_string(index=False))


   Dataset                     Display_Name                      Role  Rows  Raw_Columns      Target                                                           SHA256
       ibm                 IBM HR Analytics supplementary HR analysis  1470           35   Attrition a5c31e38bd7fafc9bc333884eb181b06b41b8e5e488e8f7ccb27199fb3be7659
job_change                       Job Change supplementary HR analysis 19158           14      target 8b78da3482032500df40d5359c36ba4a59e28ccd6fc272b050dcf6dee37f114c
 promotion               Employee Promotion supplementary HR analysis 54808           14 is_promoted 3d7b679b3ba36d1cad25d4e0ea40bc84e5ac741c45e9c878d72adc3db74a984c
      ghrm GHRM - Environmental Performance   main Green HRM analysis   320           33   FEP items ac5c0005d3492aa664ec4d290e027f04a778241d3ce6195e10c5f87aebc65203


## ۲. مدل‌های طبقه‌بندی و مدل منتخب هر مسئله


In [3]:
classification = pd.read_csv(RESULTS / "tables" / "classification_metrics.csv")
best_classifiers = classification.loc[classification.groupby("Dataset")["F1"].idxmax(), ["Dataset", "Model", "Accuracy", "Precision", "Recall", "F1", "CV_Best_F1"]]
print(best_classifiers.to_string(index=False))


   Dataset         Model  Accuracy  Precision   Recall       F1  CV_Best_F1
       ibm           MLP  0.877551   0.703704 0.404255 0.513514    0.537699
job_change Random Forest  0.762004   0.515523 0.747644 0.610256    0.584057
 promotion           MLP  0.943076   0.972561 0.341542 0.505547    0.494707


## ۳. تحلیل اصلی GHRM و پیش‌بینی FEP


In [4]:
regression = pd.read_csv(RESULTS / "tables" / "regression_metrics.csv")
columns = ["Variant", "Model", "R2", "RMSE", "MAE", "R2_CI_Lower", "R2_CI_Upper", "Repeated_CV_R2_Mean", "Tuning_Convergence_Warnings", "Repeated_CV_Convergence_Warnings"]
print(regression[columns].to_string(index=False))


Variant                   Model       R2     RMSE      MAE  R2_CI_Lower  R2_CI_Upper  Repeated_CV_R2_Mean  Tuning_Convergence_Warnings  Repeated_CV_Convergence_Warnings
   Base Random Forest Regressor 0.384701 0.509342 0.390547     0.053830     0.594363             0.415293                            0                                 0
   Base Decision Tree Regressor 0.422155 0.493596 0.383052     0.041499     0.652477             0.368388                            0                                 0
   Base               LinearSVR 0.428678 0.490802 0.383129     0.146233     0.594364             0.465326                            0                                 0
   Base            MLPRegressor 0.371678 0.514704 0.406078     0.098728     0.531163             0.301354                            0                                 0
   GEE+ Random Forest Regressor 0.398212 0.503719 0.390486     0.070613     0.602281             0.425657                            0                     

## ۴. اثر پیش‌بینانه افزودن GEE


In [5]:
comparison = pd.read_csv(RESULTS / "tables" / "base_vs_gee_plus.csv")
print(comparison.to_string(index=False))


                  Model  Delta_R2  Delta_R2_CI_Lower  Delta_R2_CI_Upper  Delta_MAE  Delta_MAE_CI_Lower  Delta_MAE_CI_Upper  Delta_RMSE  Delta_RMSE_CI_Lower  Delta_RMSE_CI_Upper
Random Forest Regressor  0.013511          -0.030336           0.059500  -0.000061           -0.014813            0.015301   -0.005623            -0.023947             0.011295
Decision Tree Regressor -0.053005          -0.233573           0.159872   0.022219           -0.033442            0.084040    0.022142            -0.056465             0.113467
              LinearSVR -0.022897          -0.079731           0.010784   0.005311           -0.006894            0.018126    0.009738            -0.005747             0.025043
           MLPRegressor -0.029471          -0.153897           0.116198   0.011910           -0.027707            0.053124    0.011933            -0.040349             0.067595


## ۵. انتخاب تعداد خوشه‌ها


In [6]:
kmeans = pd.read_csv(RESULTS / "tables" / "kmeans_selection.csv")
print(kmeans[["Dataset", "k", "Numeric_Features", "Inertia_SSE", "Silhouette", "Davies_Bouldin", "Interpretation"]].to_string(index=False))


   Dataset  k  Numeric_Features   Inertia_SSE  Silhouette  Davies_Bouldin                 Interpretation
       ibm  2                23  29253.107941    0.152859        2.382461     تفکیک ضعیف و صرفاً اکتشافی
job_change  3                 2  11219.406410    0.577293        0.663936 تفکیک نسبتاً روشن در همین داده
 promotion  5                 7 191896.698482    0.256547        1.269543          تفکیک متوسط و اکتشافی
      ghrm  2                 5    770.645339    0.436339        0.861054          تفکیک متوسط و اکتشافی


## ۶. الگوی پنهان مستقیم GHRM

در این بخش خوشه‌ها فقط با پنج سازه سبز ساخته شده‌اند. `FEP` در تشکیل خوشه دخالت نداشته و بعد از آن برای توصیف عملکرد محیط‌زیستی هر خوشه اضافه شده است.


In [7]:
patterns = pd.read_csv(RESULTS / "clustering" / "ghrm" / "cluster_distinguishing_features.csv")
targets = pd.read_csv(RESULTS / "clustering" / "ghrm" / "cluster_target_summary.csv")
print("ویژگی‌های متمایزکننده:")
print(patterns.to_string(index=False))
print("\nعملکرد محیط‌زیستی پس از خوشه‌بندی:")
print(targets.to_string(index=False))


ویژگی‌های متمایزکننده:
Dataset  Cluster Feature  Cluster_Mean  Overall_Mean  Standardized_Difference Direction
   ghrm        0     GPA      3.968013      3.455729                 0.591452    higher
   ghrm        0     GEE      3.925505      3.427344                 0.572256    higher
   ghrm        0     GCM      3.954545      3.453125                 0.570962    higher
   ghrm        0     GTD      3.931313      3.466250                 0.556210    higher
   ghrm        0     GRS      3.916667      3.442188                 0.533166    higher
   ghrm        1     GPA      2.624317      3.455729                -0.959898     lower
   ghrm        1     GEE      2.618852      3.427344                -0.928744     lower
   ghrm        1     GCM      2.639344      3.453125                -0.926643     lower
   ghrm        1     GTD      2.711475      3.466250                -0.902702     lower
   ghrm        1     GRS      2.672131      3.442188                -0.865303     lower

عملکرد م

## ۷. کنترل مستقل فایل‌های منتشرشده


In [8]:
validation = subprocess.run([sys.executable, "scripts/validate_results.py"], cwd=ROOT, check=True, capture_output=True, text=True)
print(validation.stdout.strip())


validation passed: 4 datasets, 12 classifiers, 8 regressors, 4 clustering runs
